## Check the setup and connect to the database

In [17]:
%run "../01-check_setup.ipynb"

SAP HANA Client for Python: 2.28.26031701
Using the dot-env file /home/user/projects/hana-ai-ve-kg-codejam/.env
Connected to SAP HANA db version 4.00.000.00.1774346768 (fa/CE2026.2) 
at c5889dd5-e0f6-4930-8408-94d53ca61dbf.hna0.prod-us10.hanacloud.ondemand.com:443 as DBADMIN_VR
Current time on the SAP HANA server: 2026-04-03 14:58:19.556000


In [18]:
myconn.get_tables(schema='NHTSA')

,TABLE_NAME
0,COMPLAINTS


In [3]:
import pandas as pd

Loading data from https://www.nhtsa.gov/nhtsa-datasets-and-apis#complaints

In [4]:
import_df = pd.read_csv('https://static.nhtsa.gov/odi/ffdd/cmpl/COMPLAINTS_RECEIVED_2025-2026.zip', sep='\t', header=None)
import_df.shape

/tmp/ipykernel_23513/1324197279.py:1: DtypeWarning: Columns (29,41,44) have mixed types. Specify dtype option on import or set low_memory=False.
  import_df = pd.read_csv('https://static.nhtsa.gov/odi/ffdd/cmpl/COMPLAINTS_RECEIVED_2025-2026.zip', sep='\t', header=None)


(140496, 49)

In [5]:
import_df.rename(columns={
    0: 'CMPLID',
    1: 'ODINO',
    2: 'MFR_NAME',
    3: 'MAKETXT',
    4: 'MODELTXT',
    5: 'YEARTXT',
    6: 'CRASH',
    7: 'FAILDATE',
    8: 'FIRE',
    9: 'INJURED',
    10: 'DEATHS',
    11: 'COMPDESC',
    12: 'CITY',
    13: 'STATE',
    14: 'VIN',
    15: 'DATEA',
    16: 'LDATE',
    17: 'MILES',
    18: 'OCCURENCES',
    19: 'CDESCR',
    20: 'CMPL_TYPE',
    21: 'POLICE_RPT_YN',
    22: 'PURCH_DT',
    23: 'ORIG_OWNER_YN',
    24: 'ANTI_BRAKES_YN',
    25: 'CRUISE_CONT_YN',
    26: 'NUM_CYLS',
    27: 'DRIVE_TRAIN',
    28: 'FUEL_SYS',
    29: 'FUEL_TYPE',
    30: 'TRANS_TYPE',
    31: 'VEH_SPEED',
    32: 'DOT',
    33: 'TIRE_SIZE',
    34: 'LOC_OF_TIRE',
    35: 'TIRE_FAIL_TYPE',
    36: 'ORIG_EQUIP_YN',
    37: 'MANUF_DT',
    38: 'SEAT_TYPE',
    39: 'RESTRAINT_TYPE',
    40: 'DEALER_NAME',
    41: 'DEALER_TEL',
    42: 'DEALER_CITY',
    43: 'DEALER_STATE',
    44: 'DEALER_ZIP',
    45: 'PROD_TYPE',
    46: 'REPAIRED_YN',
    47: 'MEDICAL_ATTN',
    48: 'VEHICLES_TOWED_YN'
}, inplace=True)

In [6]:
# Combine them into a single DataFrame
pd.DataFrame({
    'Original Dtypes': import_df.dtypes,
    'Converted Dtypes': import_df.convert_dtypes().dtypes
})

,Original Dtypes,Converted Dtypes
CMPLID,int64,Int64
ODINO,int64,Int64
MFR_NAME,object,string[python]
MAKETXT,object,string[python]
MODELTXT,object,string[python]
YEARTXT,int64,Int64
CRASH,object,string[python]
FAILDATE,int64,Int64
FIRE,object,string[python]
INJURED,int64,Int64


In [7]:
import_df=import_df.convert_dtypes()

In [8]:
import_df[import_df['CDESCR'].str.len() > 2048][['CMPLID']].assign(CDESCR_LEN=import_df['CDESCR'].str.len())

,CMPLID,CDESCR_LEN
57206,2109516,3293
57232,2109543,4139
57233,2109545,19902
60668,2112997,3626
60669,2112999,3746
118317,2170651,10437
121841,2174185,3254
127930,2180275,3709
127931,2180277,37021
134325,2186710,9535


In [15]:
pd.set_option('max_colwidth', None) 
display(
    import_df
    .head(1).T
    .style.set_properties(subset=[0], **{'text-align': 'left'})
)

,0
CMPLID,2052309
ODINO,11633472
MFR_NAME,Ford Motor Company
MAKETXT,FORD
MODELTXT,EXPLORER
YEARTXT,2016
CRASH,N
FAILDATE,20240701
FIRE,N
INJURED,0


In [19]:
carcomplaints_hdf = hana_ml.dataframe.create_dataframe_from_pandas(myconn,
                                           import_df[import_df['CDESCR'].str.len() <= 2048],
                                            table_name="COMPLAINTS_2025_2026",
                                            schema='NHTSA',
                                            force=True)

100%|██████████| 3/3 [00:06<00:00,  2.11s/it]


In [20]:
carcomplaints_hdf.shape

[140470, 49]

In [21]:
myconn.get_tables(schema="NHTSA")

,TABLE_NAME
0,COMPLAINTS
1,COMPLAINTS_2025_2026


In [26]:
myconn.table("COMPLAINTS_2025_2026", schema='NHTSA').describe().collect() #.sort_values(by='unique', ascending=True)

,column,count,unique,nulls,mean,std,min,max,median,25_percent_cont,25_percent_disc,50_percent_cont,50_percent_disc,75_percent_cont,75_percent_disc
0,CMPLID,140470,140470,0,2.122579e+06,40575.815861,2052309.0,2192899.0,2122585.0,2087437.25,2087437.0,2122584.5,2122584.0,2157702.75,2157703.0
1,ODINO,140470,94765,0,1.168155e+07,27466.986489,11633472.0,11728684.0,11682035.0,11657766.00,11657766.0,11682035.0,11682035.0,11705279.00,11705279.0
2,YEARTXT,140470,35,0,2.131855e+03,940.931041,1986.0,9999.0,2020.0,2017.00,2017.0,2020.0,2020.0,2023.00,2023.0
3,FAILDATE,140470,2479,0,2.024668e+07,35680.821066,19541125.0,20260401.0,20250619.0,20250205.00,20250205.0,20250619.0,20250619.0,20251023.75,20251024.0
4,INJURED,140470,13,0,3.400014e-02,0.389527,0.0,99.0,0.0,0.00,0.0,0.0,0.0,0.00,0.0
5,DEATHS,140470,7,0,1.523457e-03,0.095147,0.0,15.0,0.0,0.00,0.0,0.0,0.0,0.00,0.0
6,DATEA,140470,455,0,2.025276e+07,3961.858768,20250101.0,20260401.0,20250821.0,20250430.00,20250430.0,20250821.0,20250821.0,20251216.00,20251216.0
7,LDATE,140470,455,0,2.025276e+07,3961.858768,20250101.0,20260401.0,20250821.0,20250430.00,20250430.0,20250821.0,20250821.0,20251216.00,20251216.0
8,MILES,22223,4011,118247,8.125617e+04,59869.578775,0.0,880000.0,75450.0,37000.00,37000.0,75450.0,75450.0,115000.00,115000.0
9,OCCURENCES,0,0,140470,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
